Import Library

In [48]:
import pandas as pd
import numpy as np

from prophet import Prophet

from sqlalchemy import create_engine
from sqlalchemy import text

In [49]:
#koneksi laravel
engine = create_engine("mysql+pymysql://root:@localhost/pos_ta")
print("Koneksi ke database berhasil")

Koneksi ke database berhasil


In [50]:
#file excel
FILE_EXCEL = "dataset/data_clean.xlsx"

In [51]:
hasil_semua = []

for sheet in sheet_produk:
    print(f"Proses {sheet}")

    # DATA PENJUALAN
    df = pd.read_excel(FILE_EXCEL, sheet_name=sheet)
    df['ds'] = pd.to_datetime(df['ds'])

    # PROPHET
    model = Prophet()
    model.fit(df)
    future = model.make_future_dataframe(periods=14)
    forecast = model.predict(future)
    forecast_14 = forecast.tail(14)
    forecast_qty = int(round(forecast_14['yhat'].sum()))

    # PRODUCT ID
    product_id = mapping_produk[sheet]

    # STOK SEKARANG
    query_stock = f"""
    SELECT current_stock
    FROM stocks
    WHERE product_id = {product_id}
    """

    stok = pd.read_sql(query_stock, engine)
    if len(stok) == 0:
        current_stock = 0
    else:
        current_stock = int(
            stok.iloc[0]['current_stock']
        )

    # SUPPLIER
    query_supplier = f"""
    SELECT
        s.lead_time,
        s.MOQ
    FROM product_suppliers ps
    JOIN suppliers s
        ON ps.supplier_id = s.id
    WHERE ps.product_id = {product_id}
    """

    supplier = pd.read_sql(query_supplier, engine)
    if len(supplier) == 0:
        lead_time = 14
        moq = 1
    else:
        lead_time = int(supplier.iloc[0]['lead_time'])
        moq = int(supplier.iloc[0]['MOQ'])

    # SAFETY STOCK
    std_harian = df['y'].std()
    safety_stock = int(round(1.65 * std_harian * np.sqrt(lead_time)))

    # PERMINTAAN HARIAN
    daily_demand = (forecast_qty / 14)

    # ROP
    rop = int(round((daily_demand * lead_time) + safety_stock))

    # REKOMENDASI BELI
    rekomendasi = max(0, rop - current_stock)

    # MOQ
    if rekomendasi > 0:
        if rekomendasi < moq: rekomendasi = moq
        else:
            rekomendasi = (
                int(np.ceil(rekomendasi / moq)) * moq
                )

    hasil_semua.append({
        "product_id": product_id,
        "produk": sheet,
        "forecast_qty": forecast_qty,
        "lead_time": lead_time,
        "moq": moq,
        "current_stock": current_stock,
        "daily_demand": round(daily_demand, 2),
        "safety_stock": safety_stock,
        "rop": rop,
        "rekomendasi": rekomendasi
    })

Proses surya_kaleng


23:29:18 - cmdstanpy - INFO - Chain [1] start processing
23:29:19 - cmdstanpy - INFO - Chain [1] done processing


Proses sampoerna_16


23:29:19 - cmdstanpy - INFO - Chain [1] start processing
23:29:19 - cmdstanpy - INFO - Chain [1] done processing


Proses aqua_600ml


23:29:20 - cmdstanpy - INFO - Chain [1] start processing
23:29:20 - cmdstanpy - INFO - Chain [1] done processing


Proses aqua_1500ml


23:29:21 - cmdstanpy - INFO - Chain [1] start processing
23:29:21 - cmdstanpy - INFO - Chain [1] done processing


Proses indomie


23:29:21 - cmdstanpy - INFO - Chain [1] start processing
23:29:21 - cmdstanpy - INFO - Chain [1] done processing


In [52]:
hasil_df = pd.DataFrame(
    hasil_semua
)

hasil_df

,product_id,produk,forecast_qty,lead_time,moq,current_stock,daily_demand,safety_stock,rop,rekomendasi
0,11,surya_kaleng,1508,14,20,130,107.71,272,1780,1660
1,12,sampoerna_16,217,14,20,100,15.50,44,261,180
2,10,aqua_600ml,155,14,50,27,11.07,59,214,200
3,9,aqua_1500ml,180,14,50,6,12.86,32,212,250
4,13,indomie,159,14,24,150,11.36,33,192,48


In [53]:
#Simpan Forecast
with engine.begin() as conn:
    conn.execute(
        text("DELETE FROM forecast_results")
    )

    for _, row in hasil_df.iterrows():
        conn.execute(
            text("""
                INSERT INTO forecast_results
                (
                    product_id,
                    forecast_period,
                    predicted_qty,
                    created_at,
                    updated_at
                )
                VALUES
                (
                    :product_id,
                    CURDATE(),
                    :predicted_qty,
                    NOW(),
                    NOW()
                )
            """),
            {
                "product_id": int(row['product_id']),
                "predicted_qty": int(row['forecast_qty'])
            }
        )

print("Forecast berhasil disimpan")

Forecast berhasil disimpan


In [54]:
#Simpan Purchase Recommendation
with engine.begin() as conn:
    conn.execute(
        text("DELETE FROM purchase_recommendations")
    )

    for _, row in hasil_df.iterrows():
        conn.execute(
            text("""
                INSERT INTO purchase_recommendations
                (
                    product_id,
                    recommendation_qty,
                    created_at,
                    updated_at
                )
                VALUES
                (
                    :product_id,
                    :recommendation_qty,
                    NOW(),
                    NOW()
                )
            """),
            {
                "product_id": int(row['product_id']),
                "recommendation_qty": int(row['rekomendasi'])
            }
        )

print("Purchase Recommendation berhasil disimpan")

Purchase Recommendation berhasil disimpan


In [55]:
#update stok
with engine.begin() as conn:
    for _, row in hasil_df.iterrows():
        conn.execute(
            text("""
                UPDATE stocks
                SET
                    safety_stock = :safety_stock,
                    reorder_point = :rop,
                    updated_at = NOW()
                WHERE product_id = :product_id
            """),
            {
                "safety_stock": int(row['safety_stock']),
                "rop": int(row['rop']),
                "product_id":int(row['product_id'])
            }
        )

print("Stock berhasil diperbarui")

Stock berhasil diperbarui
